In [ ]:
# import libraries
try:
  # %tensorflow_version only exists in Colab.
  !pip install tf-nightly
except Exception:
  pass
import tensorflow as tf
import pandas as pd
from tensorflow import keras
!pip install tensorflow-datasets
import tensorflow_datasets as tfds
import numpy as np
import matplotlib.pyplot as plt

print(tf.__version__)

In [ ]:
# get data files
!wget https://cdn.freecodecamp.org/project-data/sms/train-data.tsv
!wget https://cdn.freecodecamp.org/project-data/sms/valid-data.tsv

train_file_path = "train-data.tsv"
test_file_path = "valid-data.tsv"

In [ ]:
# Load the tab-separated SMS datasets.
train_df = pd.read_csv(
    train_file_path,
    sep='\t',
    header=None,
    names=['label', 'message'],
)

test_df = pd.read_csv(
    test_file_path,
    sep='\t',
    header=None,
    names=['label', 'message'],
)

# Encode labels as required by the challenge: ham=0, spam=1.
label_map = {'ham': 0, 'spam': 1}

train_texts = train_df['message'].astype(str).values
train_labels = train_df['label'].map(label_map).astype('float32').values

test_texts = test_df['message'].astype(str).values
test_labels = test_df['label'].map(label_map).astype('float32').values

print(train_df['label'].value_counts())
print(test_df['label'].value_counts())

In [ ]:
# Build an end-to-end neural text classifier.
tf.keras.utils.set_random_seed(42)

max_tokens = 15000
sequence_length = 100

vectorizer = keras.layers.TextVectorization(
    max_tokens=max_tokens,
    standardize='lower_and_strip_punctuation',
    split='whitespace',
    output_mode='int',
    output_sequence_length=sequence_length,
)
vectorizer.adapt(train_texts)

vocab_size = len(vectorizer.get_vocabulary())

model = keras.Sequential([
    keras.layers.Input(shape=(), dtype=tf.string),
    vectorizer,
    keras.layers.Embedding(
        input_dim=vocab_size,
        output_dim=64,
        mask_zero=True,
    ),
    keras.layers.Bidirectional(keras.layers.LSTM(32)),
    keras.layers.Dropout(0.35),
    keras.layers.Dense(32, activation='relu'),
    keras.layers.Dropout(0.25),
    keras.layers.Dense(1, activation='sigmoid'),
])

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='binary_crossentropy',
    metrics=[
        'accuracy',
        keras.metrics.Precision(name='precision'),
        keras.metrics.Recall(name='recall'),
    ],
)

# Compensate for the strong ham/spam class imbalance in the training set.
class_counts = np.bincount(train_labels.astype('int32'))
total = class_counts.sum()
class_weight = {
    0: total / (2.0 * class_counts[0]),
    1: total / (2.0 * class_counts[1]),
}

early_stopping = keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True,
)

history = model.fit(
    train_texts,
    train_labels,
    validation_split=0.2,
    epochs=15,
    batch_size=64,
    class_weight=class_weight,
    callbacks=[early_stopping],
    verbose=1,
)

model.evaluate(test_texts, test_labels, verbose=2)
model.summary()

In [ ]:
# function to predict messages based on model
# (should return list containing prediction and label, ex. [0.008318834938108921, 'ham'])
def predict_message(pred_text):
  probability = float(
      model.predict(tf.constant([pred_text]), verbose=0)[0][0]
  )
  label = 'spam' if probability >= 0.5 else 'ham'
  prediction = [probability, label]

  return (prediction)

pred_text = "how are you doing today?"

prediction = predict_message(pred_text)
print(prediction)

In [ ]:
# Run this cell to test your function and model. Do not modify contents.
def test_predictions():
  test_messages = ["how are you doing today",
                   "sale today! to stop texts call 98912460324",
                   "i dont want to go. can we try it a different day? available sat",
                   "our new mobile video service is live. just install on your phone to start watching.",
                   "you have won £1000 cash! call to claim your prize.",
                   "i'll bring it tomorrow. don't forget the milk.",
                   "wow, is your arm alright. that happened to me one time too"
                  ]

  test_answers = ["ham", "spam", "ham", "spam", "spam", "ham", "ham"]
  passed = True

  for msg, ans in zip(test_messages, test_answers):
    prediction = predict_message(msg)
    if prediction[1] != ans:
      passed = False

  if passed:
    print("You passed the challenge. Great job!")
  else:
    print("You haven't passed yet. Keep trying.")

test_predictions()
